In [10]:
# Day 2 · vLLM on T4.  The old `pip install vllm==0.7.3` failed because it fought Kaggle's
# torch 2.10 / transformers 5.0. vLLM now lives in its own venv (scripts/vllm_setup.sh).
# Absolute paths: safe to re-run (clone the first time, pull every time after).
%cd /kaggle/working
!if [ -d 3-day-inference-challenge/.git ]; then git -C 3-day-inference-challenge pull; else git clone https://github.com/Maheendra-mj/3-day-inference-challenge; fi
%cd /kaggle/working/3-day-inference-challenge
!git log --oneline -1

Already up to date.
/kaggle/working/3-day-inference-challenge


In [ ]:
# 1) Install vLLM + infer-lab into /tmp/vllm-env (~3-5 min, once per session)
!bash scripts/vllm_setup.sh

? A virtual environment already exists at `/tmp/vllm-env`. Do you want to replace it? [y/n] › yes

hint: Use the `--clear` flag or set `UV_VENV_CLEAR=1` to skip this prompt

In [4]:
# 2) Smoke test: does vLLM run on sm_75? Which attention backend? How big is the KV cache?
!/tmp/vllm-env/bin/python scripts/vllm_smoke.py > /tmp/vllm_smoke.log 2>&1; echo "exit code: $?"
!grep -i -E "backend|kv cache|cuda graph|concurrency" /tmp/vllm_smoke.log | cut -c1-220
!tail -n 25 /tmp/vllm_smoke.log

exit code: 0
(EngineCore pid=292) INFO 10-05 09:40:55 [core.py:129] Initializing a V1 LLM engine (v0.31.0) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-Instruct', skip_tokeniz
(EngineCore pid=292) INFO 10-05 09:40:57 [parallel_state.py:1825] world_size=1 rank=0 local_rank=0 distributed_init_method=file:///tmp/vllm_dist_f8f0998bfbd54f36b2023af43cc0050f backend=nccl
(EngineCore pid=292) INFO 10-05 09:41:03 [cuda.py:526] Using TRITON_ATTN attention backend out of potential backends: ['TRITON_ATTN', 'FLEX_ATTENTION'].
(EngineCore pid=292) INFO 10-05 09:41:12 [utils.py:343] Using LBNHC KV cache layout.
(EngineCore pid=292) INFO 10-05 09:41:20 [backends.py:1090] Using cache directory: /root/.cache/vllm/torch_compile_cache/42810a1a34/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=292) INFO 10-05 09:41:20 [backends.py:1151] Dynamo bytecode transform time: 6.24 s
(EngineCore pid=292) INFO 10-05 09:41:30 [backends.py:391] Comp

In [13]:
%cd /kaggle/working
!git -C 3-day-inference-challenge pull
!ls 3-day-inference-challenge/3-day-inference-challenge && rm -rf 3-day-inference-challenge/3-day-inference-challenge
%cd /kaggle/working/3-day-inference-challenge
!git log --oneline -1


/kaggle/working
remote: Enumerating objects: 11, done.
remote: Counting objects: 100% (11/11), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 6 (delta 3), reused 6 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (6/6), 744 bytes | 372.00 KiB/s, done.
From https://github.com/Maheendra-mj/3-day-inference-challenge
   78736d1..3f245e1  main       -> origin/main
Updating 78736d1..3f245e1
Fast-forward
 src/infer_lab/telemetry/env.py | 15 ++++++++++-----
 1 file changed, 10 insertions(+), 5 deletions(-)
configs  LICENSE    pyproject.toml  scripts  tests
docs	 notebooks  README.md	    src
/kaggle/working/3-day-inference-challenge
3f245e1 (HEAD -> main, origin/main, origin/HEAD) environment


In [14]:
!/tmp/vllm-env/bin/infer-lab env

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
python      3.13.15
torch       2.13.0+cu130
transformers 5.17.0
vllm        0.31.0
onnxruntime -
tensorrt    -
triton      3.7.1
cpu         Intel(R) Xeon(R) CPU @ 2.00GHz (4 vCPU)
driver      580.178.04
cuda        13.0
gpu0        Tesla T4  sm_75  14.6 GiB  40 SMs
gpu1        Tesla T4  sm_75  14.6 GiB  40 SMs
p2p 0<->1   True


In [15]:
# 3) Only after the smoke test passes: vLLM offline, same sweep as the Day-1 baseline (CUDA graphs ON)
!/tmp/vllm-env/bin/infer-lab bench --backend vllm_offline --config configs/vllm_offline.yaml --tag vllm 2>&1 | grep -E "infer_lab|Error|Traceback"

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
2026-10-05 09:55:44,885 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on vllm_offline {'gpu_memory_utilization': 0.85, 'max_model_len': 2048, 'enforce_eager': False}
2026-10-05 09:57:01,300 INFO infer_lab.bench.runner: run 1  bs=1   pl=128   ol=128   tok/s=   173.8  ttft_p50=   18.6ms  tpot_p50=  5.64ms  util= 99%  power=68.1W  sm_clk=1164MHz
2026-10-05 09:57:04,490 INFO infer_lab.bench.runner: run 2  bs=1   pl=512   ol=128   tok/s=   161.2  ttft_p50=   44.9ms  tpot_p50=  5.89ms  util=100%  power=66.8W  sm_clk=1195MHz
2026-10-05 09:57:07,597 INFO infer_lab.bench.runner: run 3  bs=4   pl=128   ol=128   tok/s=   668.4  ttft_p50=   36.4ms  tpot_p50=  5.73ms  util= 98%  power=68.2W  sm_clk=1328MHz
2026-10-05 09:57:11,461 INFO infer_lab.bench.runner: run 4  bs=4   pl=512   ol=128   tok/s=   532.9  ttft_p50=  155.5ms  tpot_p50=  6.34ms  util= 99%  power=68.3W  sm_clk=1332MH

In [16]:
# 4) vLLM with CUDA graphs OFF: separates "launch floor removed" from "better attention kernels"
!/tmp/vllm-env/bin/infer-lab bench --backend vllm_offline --config configs/vllm_offline_eager.yaml --tag vllm_eager 2>&1 | grep -E "infer_lab|Error|Traceback"

Error in sitecustomize; set PYTHONVERBOSE for traceback:
ModuleNotFoundError: No module named 'wrapt'
2026-10-05 09:57:50,183 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on vllm_offline {'gpu_memory_utilization': 0.85, 'max_model_len': 2048, 'enforce_eager': True}
2026-10-05 09:58:24,919 INFO infer_lab.bench.runner: run 9  bs=1   pl=128   ol=128   tok/s=    37.5  ttft_p50=   55.4ms  tpot_p50= 26.69ms  util= 31%  power=40.7W  sm_clk= 585MHz
2026-10-05 09:58:38,424 INFO infer_lab.bench.runner: run 10  bs=1   pl=512   ol=128   tok/s=    38.3  ttft_p50=   83.6ms  tpot_p50= 25.49ms  util= 31%  power=41.8W  sm_clk= 759MHz
2026-10-05 09:58:53,734 INFO infer_lab.bench.runner: run 11  bs=4   pl=128   ol=128   tok/s=   146.3  ttft_p50=   80.2ms  tpot_p50= 26.88ms  util= 30%  power=40.1W  sm_clk= 648MHz
2026-10-05 09:59:08,073 INFO infer_lab.bench.runner: run 12  bs=4   pl=512   ol=128   tok/s=   143.0  ttft_p50=  213.1ms  tpot_p50= 26.64ms  util= 29%  power=50.0W  sm_clk=1284

In [17]:
# 5) HF eager baseline on THIS VM (system python, Kaggle's torch), so every row in the
#    comparison comes from the same machine: launch-bound numbers vary across VMs (DAY1 §8.6)
!pip install -q -e . && infer-lab bench --config configs/bench.yaml --tag hf_baseline 2>&1 | grep -E "infer_lab|Error|Traceback"

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for infer-lab (pyproject.toml) ... done
2026-10-05 10:01:09,034 INFO infer_lab.bench.runner: loading Qwen/Qwen2.5-0.5B-Instruct on hf_torch 
2026-10-05 10:01:31,161 INFO infer_lab.bench.runner: run 17  bs=1   pl=128   ol=128   tok/s=    33.8  ttft_p50=   32.8ms  tpot_p50= 29.55ms  util= 38%  power=38.9W  sm_clk= 913MHz
2026-10-05 10:01:46,283 INFO infer_lab.bench.runner: run 18  bs=1   pl=512   ol=128   tok/s=    33.8  ttft_p50=   46.4ms  tpot_p50= 29.46ms  util= 35%  power=48.3W  sm_clk=1328MHz
2026-10-05 10:02:01,901 INFO infer_lab.bench.runner: run 19  bs=4   pl=128   ol=128   tok/s=   130.2  ttft_p50=   34.8ms  tpot_p50= 30.69ms  util= 36%  power=46.3W  sm_clk=1263MHz
2026-10-05 10:02:18,241 INFO infer_lab.bench.runner: run 20  bs=4   pl=512   ol=128   tok/s

In [18]:
# 6) Side-by-side: hf_baseline vs vllm_eager vs vllm
!infer-lab report

 id         tag      backend  batch_size  prompt_len  output_len  output_tokens_per_s  ttft_p50_ms  ttft_p99_ms  tpot_p50_ms  e2e_p99_ms  torch_peak_mem_mb  gpu.gpu0.util_mean_pct  gpu.gpu0.power_mean_w
  1        vllm vllm_offline           1         128         128               173.76        18.57        23.42         5.64      739.10                NaN                   99.05                  68.10
  2        vllm vllm_offline           1         512         128               161.19        44.88        46.48         5.89      796.35                NaN                   99.55                  66.84
  3        vllm vllm_offline           4         128         128               668.41        36.39        40.09         5.73      768.08                NaN                   98.50                  68.21
  4        vllm vllm_offline           4         512         128               532.93       155.50       157.80         6.34      960.75                NaN                   99.28         